# Brain — Multi-Agent System

**Google Colab notebook** — clone the repo, install deps, configure the
Ollama backend with role-based model assignment, and run the Brain.

| Role | Agents | Typical model |
|------|--------|---------------|
| `purpose` | direction, memory, QA, orchestrator, validator, goal_evaluator | General-purpose LLM (e.g. `llama3.1:8b`) |
| `coder`   | action (code exec), search (tool calls) | Code-specialised LLM (e.g. `qwen2.5-coder:7b`) |

Recommended runtime: **L4 GPU** (Colab Pro) or **T4** (free tier).

## 1 — Mount Google Drive
ChromaDB and episode SQLite are stored on Drive so they survive session restarts.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib

DRIVE_DATA = '/content/drive/MyDrive/Brain/data'
pathlib.Path(DRIVE_DATA).mkdir(parents=True, exist_ok=True)

print(f'Drive mounted. Data will persist at: {DRIVE_DATA}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Data will persist at: /content/drive/MyDrive/Brain/data


## 2 — Clone the repo

In [2]:
import os

REPO_URL = 'https://github.com/Seydifa/Brain.git'
REPO_DIR = '/content/Brain'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print('Working directory:', os.getcwd())

Already up to date.
/content/Brain
Working directory: /content/Brain


## 3 — Link Drive data directory

In [3]:
import os, shutil

REPO_DATA  = '/content/Brain/data'
DRIVE_DATA = '/content/drive/MyDrive/Brain/data'

if os.path.exists(REPO_DATA) and not os.path.islink(REPO_DATA):
    shutil.rmtree(REPO_DATA)
if os.path.islink(REPO_DATA):
    os.remove(REPO_DATA)

os.symlink(DRIVE_DATA, REPO_DATA)
print(f'data/ -> {DRIVE_DATA}')

data/ -> /content/drive/MyDrive/Brain/data


## 4 — Install dependencies

In [4]:
!pip install -q \
    langgraph \
    langgraph-checkpoint-sqlite \
    langchain-google-genai \
    langchain-chroma \
    langchain-ollama \
    ddgs \
    httpx \
    python-dotenv

print('Dependencies installed.')

Dependencies installed.


## 5 — Backend & Role Configuration

Set the backend and assign a model to each role.  
`OLLAMA_MAX_LOADED_MODELS` is auto-sized to the number of unique models.

To switch to **Gemini** instead, change `BACKEND` to `'gemini'`,
set `ROLES` values to `None`, and make sure `GOOGLE_API_KEY` is set.

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
#  Backend + Role → Model mapping
# ══════════════════════════════════════════════════════════════════════════════

BACKEND = 'ollama'   # 'gemini' | 'ollama'

ROLES = {
    'purpose': 'llama3.1:8b',         # general reasoning  (direction, memory, QA, …)
    'coder':   'llama3.1:8b'#'qwen2.5-coder:7b',    # tool-call agents   (action, search)
}

# ── Gemini alternative ──
# BACKEND = 'gemini'
# ROLES   = {'purpose': None, 'coder': None}   # all roles share gemini-2.5-pro

# ══════════════════════════════════════════════════════════════════════════════
import os, sys, subprocess, time, shutil

if BACKEND == 'gemini':
    # ── Gemini: load API key ───────────────────────────────────────────
    from google.colab import userdata
    try:
        os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
        print('Key loaded from Colab Secrets.')
    except Exception:
        os.environ['GOOGLE_API_KEY'] = 'YOUR_GEMINI_API_KEY_HERE'
        print('Key set inline — update the value above.')
    with open('/content/Brain/.env', 'w') as f:
        f.write(f"GOOGLE_API_KEY={os.environ['GOOGLE_API_KEY']}\n")
    print(f'Backend : gemini')
    print(f'Key     : ...{os.environ["GOOGLE_API_KEY"][-6:]}')

else:
    # ── Ollama: install + serve + pull models ──────────────────────────
    os.environ['PATH'] = '/usr/local/bin:' + os.environ.get('PATH', '')
    os.system('apt-get install -y -q zstd')
    os.system('curl -fsSL https://ollama.com/install.sh | sh')

    ollama_bin = shutil.which('ollama') or '/usr/local/bin/ollama'
    if not os.path.isfile(ollama_bin):
        raise RuntimeError('Ollama install failed — binary not found.')

    subprocess.Popen([ollama_bin, 'serve'],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(3)

    # Collect every unique model (roles + embeddings)
    EMBED_MODEL = 'nomic-embed-text'
    models_to_pull = {EMBED_MODEL}
    default_model = ROLES.get('purpose') or 'llama3.1:8b'
    models_to_pull.add(default_model)
    for m in ROLES.values():
        if m:
            models_to_pull.add(m)

    for model in sorted(models_to_pull):
        print(f'Pulling {model}…')
        os.system(f'ollama pull {model}')

    print(f'\nBackend : ollama')
    print(f'Models  : {sorted(models_to_pull)}')

os.environ['BRAIN_BACKEND'] = BACKEND
print(f'Roles   : {ROLES}')

Pulling llama3.1:8b…
Pulling nomic-embed-text…

Backend : ollama
Models  : ['llama3.1:8b', 'nomic-embed-text']
Roles   : {'purpose': 'llama3.1:8b', 'coder': 'llama3.1:8b'}


## 6 — Load the graph

In [ ]:
import subprocess, sys, uuid

# Force-sync to latest commit
subprocess.run(['git', '-C', '/content/Brain', 'fetch', 'origin', 'main'],
               capture_output=True)
reset = subprocess.run(['git', '-C', '/content/Brain', 'reset', '--hard', 'origin/main'],
                       capture_output=True, text=True)
print('git sync:', reset.stdout.strip() or reset.stderr.strip()[:120])

# Clear bytecode + module caches
subprocess.run(['find', '/content/Brain', '-name', '*.pyc', '-delete'],
               capture_output=True)
for m in [k for k in sys.modules
          if k.startswith(('core', 'agents', 'memory', 'prompts'))]:
    del sys.modules[m]

sys.path.insert(0, '/content/Brain')

from dotenv import load_dotenv
load_dotenv()

# ── Configure roles BEFORE importing agents ──────────────────────────
from core.config import configure, describe

configure(backend=BACKEND, roles=ROLES)
print(describe())
print()

# ── Build graph ──────────────────────────────────────────────────────
from core.graph import get_graph

graph     = get_graph()
thread_id = str(uuid.uuid4())
config    = {'configurable': {'thread_id': thread_id}}

EMPTY_STATE = {
    'goal': '',
    'messages': [],
    'response': '',
    'status': 'empty',
    'direction_result': {},
    'action_result': {},
    'action_scratch': [],
    'action_attempts': 0,
    'oriented_context': {},
    'reasoning_trace': [],
    'retry_count': 0,
    'search_valid': False,
    'search_feedback': '',
    'qa_draft': '',
    'qa_approved': False,
    'qa_feedback': '',
    'qa_attempts': 0,
    'needs_clarification': False,
    'clarification_reason': '',
    'clarification_questions': [],
}

print('Graph loaded. Ready to ask questions.')

git sync: HEAD is now at 967b4f1 chore: restore notebook to pre-edit state
backend : ollama
host    : http://localhost:11434
embed   : nomic-embed-text
default : llama3.1:8b

Role assignments:
  coder      → llama3.1:8b (default)  [action, search]
  purpose    → llama3.1:8b (default)  [direction, memory_classify, memory_store, search_validator, orchestrator, qa, goal_evaluator]

Unique models: 1  ['llama3.1:8b']
OLLAMA_MAX_LOADED_MODELS: 2

Graph loaded. Ready to ask questions.


## 7 — Verify configuration

In [7]:
import subprocess, shutil
import core.state

# ── Runtime constants ────────────────────────────────────────────────
print('Runtime constants:')
print(f'  MEMORY_SCORE_THRESHOLD : {core.state.MEMORY_SCORE_THRESHOLD}')
print(f'  MAX_SEARCH_RETRIES     : {core.state.MAX_SEARCH_RETRIES}')
print(f'  MAX_QA_ATTEMPTS        : {core.state.MAX_QA_ATTEMPTS}')
print(f'  MAX_ACTION_RETRIES     : {core.state.MAX_ACTION_RETRIES}')

# ── Git head ─────────────────────────────────────────────────────────
log = subprocess.run(['git', '-C', '/content/Brain', 'log', '--oneline', '-4'],
                     capture_output=True, text=True)
print(f'\nRecent commits:\n{log.stdout.strip()}')

# ── Active models + search tools ─────────────────────────────────────
from agents.search_agent import (
    web_search_tool, academic_search_tool, wikipedia_tool, arxiv_tool, _llm,
)
print(f'\nSearch LLM model: {getattr(_llm, "model", "—")}')
print('Search tools:')
for t in [web_search_tool, academic_search_tool, wikipedia_tool, arxiv_tool]:
    print(f'  ✓ {t.name}')

# ── GPU VRAM (if available) ──────────────────────────────────────────
try:
    out = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,memory.used,memory.free,memory.total',
         '--format=csv,noheader,nounits'], text=True).strip()
    print('\nGPU VRAM:')
    for line in out.splitlines():
        name, used, free, total = [s.strip() for s in line.split(',')]
        print(f'  {name}  used={used} MB  free={free} MB  total={total} MB')
except Exception:
    pass

# ── Ollama loaded models (if active) ─────────────────────────────────
ollama_bin = shutil.which('ollama') or '/usr/local/bin/ollama'
if os.path.isfile(ollama_bin):
    result = subprocess.run([ollama_bin, 'ps'], capture_output=True, text=True)
    print(f'\nOllama loaded models:\n{result.stdout or "  (none)"}')

Runtime constants:
  MEMORY_SCORE_THRESHOLD : 0.65
  MAX_SEARCH_RETRIES     : 3
  MAX_QA_ATTEMPTS        : 2
  MAX_ACTION_RETRIES     : 5

Recent commits:
967b4f1 chore: restore notebook to pre-edit state
c56c5e1 fix: action agent mandates execute_python tool call + Python language rule + isolated recursion config
6ae5f8d refactor: clean notebook — Ollama default, defined roles, reorganized sections
eb15d78 feat: role-based configure() — per-agent model assignment (purpose/coder)

Search LLM model: llama3.1:8b
Search tools:
  ✓ web_search_tool
  ✓ academic_search_tool
  ✓ wikipedia_tool
  ✓ arxiv_tool

GPU VRAM:
  NVIDIA L4  used=0 MB  free=22564 MB  total=23034 MB

Ollama loaded models:
NAME    ID    SIZE    PROCESSOR    CONTEXT    UNTIL 



## 8 — Ask the Brain
Change `goal` and re-run this cell for each question.

In [8]:
import time
from IPython.display import display, Markdown

goal = "What caused World War 2?"   # <-- change this

state = {**EMPTY_STATE, 'goal': goal}

for attempt in range(4):
    try:
        result = graph.invoke(state, config=config)
        break
    except Exception as e:
        if '429' in str(e) or 'RESOURCE_EXHAUSTED' in str(e):
            wait = 15 * (attempt + 1)
            print(f'Rate limit — retrying in {wait}s...')
            time.sleep(wait)
        else:
            raise

if result.get('needs_clarification'):
    reason    = result.get('clarification_reason', '')
    questions = result.get('clarification_questions', [])
    why_block = f'\n> **Why:** {reason}\n' if reason else ''
    questions_md = '\n'.join(f'- {q}' for q in questions)
    display(Markdown(f"### 🟡 Clarification needed\n{why_block}\n{questions_md}"))
else:
    display(Markdown(f"### 🟢 Brain Response\n\n{result.get('response', '*(no response)*')}"))

### 🟢 Brain Response

**The complex causes of World War 2**

The outbreak of World War 2 was a result of multiple factors, including [1] aggressive territorial expansion by Germany, Japan, and Italy, as well as the policy of appeasement by the Allies, which allowed Hitler to pursue his aggressive foreign policy without facing significant opposition. The Treaty of Versailles, imposed after World War 1, created a sense of resentment among the German people, contributing to [2] Hitler's aggressive policies. Additionally, the economic crisis of the Great Depression led to widespread poverty and desperation in many countries, making them more susceptible to extremist ideologies like fascism and nationalism.

The failure of the League of Nations to prevent aggression by Japan in China and Italy in Ethiopia further exacerbated tensions, while the Nazi-Soviet Pact between Germany and the Soviet Union allowed Hitler to pursue his aggressive policies without opposition from the Soviet Union [3]. The combination of these factors ultimately led to the outbreak of World War 2 in September 1939.

**Sources:**

* "Causes of World War II" by Wikipedia
* "The Causes of WWII - World History Encyclopedia"
* "The Causes of the Second World War" by GCSE History
* "Cause and Effect: The Outbreak of World War II | TeachingHistory.org"
* "World War II: Causes and Timeline | HISTORY"

Note: This summary is based on the information gathered from the provided sources, but it does not provide a clear, direct answer to the question of what caused World War 2.

## 9 — Debug: reasoning trace & episode history

In [9]:
ctx   = result.get('oriented_context', {})
trace = result.get('reasoning_trace', [])

print(f'Turn type  : {ctx.get("turn_type", "?")}')
print(f'Coverage   : {ctx.get("coverage", "?")}')
print(f'Confidence : {ctx.get("knowledge_confidence", 0):.2f}')
print(f'Episode    : {ctx.get("current_episode_id", "?")}')
print()
print('Reasoning trace:')
for i, step in enumerate(trace, 1):
    print(f'  {i}. {step}')

Turn type  : new_topic
Coverage   : full
Confidence : 0.79
Episode    : ep_9b0c304c_20260413180947

Reasoning trace:
  1. direction: new_topic | method=empty_history | sim=0.000 | bridge=no
  2. classified as new_topic | coverage=full | parent=None
  3. qa draft generated | turn=new_topic | feedback=no
  4. qa scored 8/10 | approved=True
  5. qa draft approved, promoted to final response
  6. episode finalized and stored


In [10]:
from memory.episodes import get_recent
import json

episodes = get_recent(10)
for ep in episodes:
    flags = json.loads(ep.get('flags') or '[]')
    print(f"[{ep['id']}]")
    print(f"  Request  : {ep['user_request'][:80]}")
    print(f"  Type     : {ep['turn_type']}")
    print(f"  Flags    : {flags}")
    print(f"  Response : {str(ep.get('chosen_response', ''))[:100]}...")
    print()

[ep_9b0c304c_20260413180947]
  Request  : What caused World War 2?
  Type     : new_topic
  Flags    : []
  Response : **The complex causes of World War 2**

The outbreak of World War 2 was a result of multiple factors,...

[ep_5b3cec45_20260413175725]
  Request  : Explain the difference between a list and a tuple in Python. When should you use
  Type     : new_topic
  Flags    : ['search_used']
  Response : ...

[ep_ec650f12_20260413175717]
  Request  : Give me a step-by-step overview of how a CRISPR-Cas9 experiment is carried out i
  Type     : follow_up
  Flags    : ['follow_up']
  Response : **Callback sentence:** As we discussed earlier, CRISPR-Cas9 gene editing is a technology that allows...

[ep_4cbf335d_20260413175708]
  Request  : What are the main ethical concerns and regulatory challenges around human germli
  Type     : follow_up
  Flags    : ['follow_up']
  Response : Callback sentence: As we previously discussed, CRISPR-Cas9 is generally considered to be more effici...

[

## Test A — Direction Agent (semantic + disclaimer detection)

Isolate the Direction Agent to verify topic classification:
1. **Semantic distance** (cosine similarity threshold)
2. **Explicit disclaimers** (regex patterns like "setting X aside")

In [11]:
import uuid
from agents.direction_agent import classify_direction

dir_thread = str(uuid.uuid4())

DIRECTION_TESTS = [
    ('What is general relativity?',                       'new_topic (empty history)'),
    ('What are gravitons in this theory?',                'follow_up (same topic)'),
    ('Tell me more about spacetime curvature.',           'elaboration (deeper)'),
    ('Setting relativity aside — what is thermodynamics?','new_topic (disclaimer regex)'),
    ('Can you give me an example of entropy?',            'follow_up (thermo continues)'),
]

print('=' * 70)
print('Direction Agent — semantic distance + disclaimer detection')
print('=' * 70)

for idx, (query, expectation) in enumerate(DIRECTION_TESTS, 1):
    r = classify_direction(query, thread_id=dir_thread)
    print(f'\nQ{idx}. {query[:70]}')
    print(f'    Expected : {expectation}')
    print(f'    Got      : turn_type={r["turn_type"]}  sim={r["semantic_sim"]:.3f}  method={r["method"]}')
    if r['bridge_sentence']:
        print(f'    Bridge   : {r["bridge_sentence"][:100]}')

Direction Agent — semantic distance + disclaimer detection

Q1. What is general relativity?
    Expected : new_topic (empty history)
    Got      : turn_type=new_topic  sim=0.000  method=empty_history

Q2. What are gravitons in this theory?
    Expected : follow_up (same topic)
    Got      : turn_type=new_topic  sim=0.000  method=empty_history

Q3. Tell me more about spacetime curvature.
    Expected : elaboration (deeper)
    Got      : turn_type=new_topic  sim=0.000  method=empty_history

Q4. Setting relativity aside — what is thermodynamics?
    Expected : new_topic (disclaimer regex)
    Got      : turn_type=new_topic  sim=0.000  method=empty_history

Q5. Can you give me an example of entropy?
    Expected : follow_up (thermo continues)
    Got      : turn_type=new_topic  sim=0.000  method=empty_history


## Test B — CRISPR 4-turn conversation

Four turns on **CRISPR gene editing** — exercises academic search,
multiple QA formats, and memory continuity.

| Turn | Type | Expected QA format |
|-----:|------|-------------------|
| 1 | `new_topic` | `QUESTION` + academic sources |
| 2 | `follow_up` | `COMPARISON` table |
| 3 | `elaboration` | `QUESTION` follow-up |
| 4 | `follow_up` | `HOW-TO` numbered steps |

In [12]:
import time, uuid
from IPython.display import display, Markdown

crispr_thread = str(uuid.uuid4())
crispr_config = {'configurable': {'thread_id': crispr_thread}}

CRISPR_TURNS = [
    'How does CRISPR-Cas9 gene editing work at the molecular level?',
    'Compare CRISPR-Cas9 with older gene editing methods: TALEN and zinc finger nucleases.',
    'What are the main ethical concerns and regulatory challenges around human germline editing with CRISPR?',
    'Give me a step-by-step overview of how a CRISPR-Cas9 experiment is carried out in a lab.',
]

crispr_results = []
t_total = time.time()

for idx, goal in enumerate(CRISPR_TURNS, 1):
    t0 = time.time()
    state = {**EMPTY_STATE, 'goal': goal}

    for attempt in range(4):
        try:
            r = graph.invoke(state, config=crispr_config)
            break
        except Exception as e:
            if '429' in str(e) or 'RESOURCE_EXHAUSTED' in str(e):
                wait = 20 * (attempt + 1)
                display(Markdown(f'> ⏳ Rate limit — retrying in {wait}s…'))
                time.sleep(wait)
            else:
                raise

    elapsed   = time.time() - t0
    ctx_r     = r.get('oriented_context', {})
    turn_type = ctx_r.get('turn_type', '?')
    coverage  = ctx_r.get('coverage', '?')
    conf      = ctx_r.get('knowledge_confidence', 0)

    if r.get('needs_clarification'):
        outcome = 'CLARIFICATION'
        body = '\n'.join(f'- {q}' for q in r.get('clarification_questions', []))
    else:
        outcome = 'ANSWERED'
        body    = r.get('response', '*(no response)*')

    badge = '🟢' if outcome == 'ANSWERED' else '🟡'
    display(Markdown(f'---\n### Turn {idx}/4 {badge} {outcome}\n'
                     f'**Q:** *{goal}*\n\n{body}\n\n'
                     f'> `turn={turn_type}` | `coverage={coverage}` | '
                     f'`conf={conf:.2f}` | `{elapsed:.1f}s`'))

    crispr_results.append(dict(n=idx, outcome=outcome, turn_type=turn_type,
                               coverage=coverage, conf=f'{conf:.2f}',
                               secs=f'{elapsed:.1f}'))

total = time.time() - t_total
rows = '\n'.join(
    f"| {d['n']} | {d['outcome']:<15} | {d['turn_type']:<12} | {d['coverage']:<9} "
    f"| {d['conf']:>5} | {d['secs']:>5}s |"
    for d in crispr_results
)
display(Markdown(f'---\n## CRISPR summary — {total:.0f}s total\n\n'
                 f'| # | Outcome | Turn type | Coverage | Conf | Time |\n'
                 f'|--:|---------|-----------|----------|-----:|------|\n{rows}'))

---
### Turn 1/4 🟢 ANSWERED
**Q:** *How does CRISPR-Cas9 gene editing work at the molecular level?*

**CRISPR-Cas9 Gene Editing: A Molecular Level Explanation**

CRISPR-Cas9 gene editing works at the molecular level through a three-step process: recognition, cleavage, and repair. [3] This process involves a guide RNA (sgRNA) that recognizes a specific sequence in the DNA of interest, allowing for precise modifications to be made.

Here's how it works:

1.  **Recognition**: A designed single guide RNA (sgRNA) binds to the target sequence in the gene of interest through complementary base pairing.
2.  **Cleavage**: The Cas9 enzyme cuts the DNA at the recognized site, creating a double-stranded break.
3.  **Repair**: The cell's natural repair machinery is activated to fix the break, and during this process, the gene can be edited by introducing new genetic material.

This technology has revolutionized the field of genetics and has a wide range of applications in basic research, biotechnology, and medicine.

**Sources:**

[3] The CRISPR-Cas9 gene editing technology allows for precise modifications to an organism's DNA through a three-step process involving recognition, cleavage, and repair.

> `turn=new_topic` | `coverage=full` | `conf=0.80` | `7.2s`

---
### Turn 2/4 🟢 ANSWERED
**Q:** *Compare CRISPR-Cas9 with older gene editing methods: TALEN and zinc finger nucleases.*

Here's a comparison of CRISPR-Cas9 with older gene editing methods: TALEN and zinc finger nucleases.

| **Method** | **Mechanism of Action** | **Specificity** | **Efficiency** |
| --- | --- | --- | --- |
| CRISPR-Cas9 | Guide RNA recognizes specific sequence, Cas9 enzyme cuts DNA | High specificity due to guide RNA | Generally more efficient than ZFNs and TALENs [1] |
| TALEN | Transcription activator-like effectors (TALEs) recognize specific DNA sequences | Similar to CRISPR-Cas9 in terms of specificity | Less efficient than CRISPR-Cas9, but easier to design [2] |
| ZFN | Zinc finger proteins recognize specific DNA sequences | Lower specificity compared to CRISPR-Cas9 and TALEN | Least efficient among the three methods [1] |

Sources:

* [1]: Comparison of gene editing tools
* [2]: Advantages and disadvantages of gene editing tools

> `turn=follow_up` | `coverage=full` | `conf=0.90` | `7.6s`

---
### Turn 3/4 🟢 ANSWERED
**Q:** *What are the main ethical concerns and regulatory challenges around human germline editing with CRISPR?*

Here's a callback sentence followed by an extended answer:

The previous conversation compared CRISPR-Cas9 with older gene editing methods: TALEN and zinc finger nucleases.

Now, let's discuss the main ethical concerns and regulatory challenges around human germline editing with CRISPR. 

Human germline editing raises significant ethical concerns due to its potential for unintended consequences and unequal access [1]. The possibility of off-target effects, where unintended parts of the genome are modified, is a major concern [2]. Moreover, germline editing could exacerbate existing social inequalities if only certain groups have access to these technologies.

Regulatory challenges also surround human germline editing. In 2019, the National Academy of Sciences and the National Academy of Medicine issued a report recommending that human germline editing be prohibited for reproductive purposes until further research is conducted [3]. However, some countries like China have already begun conducting germline editing experiments on humans.

Sources:

* [1] (relevance: 0.763)
* [2] (relevance: 0.761)
* [3] (relevance: 0.74)

> `turn=follow_up` | `coverage=full` | `conf=0.76` | `9.1s`

---
### Turn 4/4 🟢 ANSWERED
**Q:** *Give me a step-by-step overview of how a CRISPR-Cas9 experiment is carried out in a lab.*

**Callback sentence:** As previously explained, CRISPR-Cas9 gene editing works at the molecular level through a three-step process involving recognition, cleavage, and repair.

Here's a step-by-step overview of how a CRISPR-Cas9 experiment is carried out in a lab:

1.  **Preparation**: Design a single guide RNA (sgRNA) that recognizes the target sequence in the gene of interest.
2.  **Plasmid preparation**: Prepare a plasmid containing the sgRNA and Cas9 enzyme, which will be used to deliver the CRISPR-Cas9 complex to the cells.
3.  **Cell culture**: Grow the cells of interest (e.g., human cells or plant cells) in a lab dish.
4.  **Transfection**: Introduce the plasmid into the cells using a transfection method (e.g., electroporation, lipofection).
5.  **Incubation**: Allow the cells to incubate for several days to allow the CRISPR-Cas9 complex to take effect.
6.  **Verification**: Use techniques such as PCR or sequencing to verify that the gene has been edited.

Note: This is a general overview of the process, and specific steps may vary depending on the experimental design and cell type being used.

> `turn=follow_up` | `coverage=full` | `conf=0.68` | `8.7s`

---
## CRISPR summary — 33s total

| # | Outcome | Turn type | Coverage | Conf | Time |
|--:|---------|-----------|----------|-----:|------|
| 1 | ANSWERED        | new_topic    | full      |  0.80 |   7.2s |
| 2 | ANSWERED        | follow_up    | full      |  0.90 |   7.6s |
| 3 | ANSWERED        | follow_up    | full      |  0.76 |   9.1s |
| 4 | ANSWERED        | follow_up    | full      |  0.68 |   8.7s |

## Test C — Action Agent stress test (12 turns)

Exercises the action agent with increasingly complex code,
scratch-pad retries, solution storage, and mixed knowledge/action turns.

In [ ]:
import time, uuid
from IPython.display import display, Markdown

action_thread = str(uuid.uuid4())
action_config = {'configurable': {'thread_id': action_thread}}

ACTION_TURNS = [
    {'goal': 'Explain the difference between a list and a tuple in Python. When should you use each?',
     'expect_action': False, 'label': 'Knowledge only (no action)'},
    {'goal': 'Check what Python version is installed, and list all numpy/scipy/pandas versions available.',
     'expect_action': True, 'label': 'check_env: runtime inspection'},
    {'goal': 'Write and run a Sieve of Eratosthenes to find primes below 10000. Print count and last 10.',
     'expect_action': True, 'label': 'run_code: Sieve of Eratosthenes'},
    {'goal': 'Use that sieve to verify Goldbach\'s conjecture for all even numbers 4–5000.',
     'expect_action': True, 'label': 'run_code: Goldbach (follow-up)'},
    {'goal': 'Create a random 100x100 matrix, compute eigenvalues, verify trace == sum of eigenvalues.',
     'expect_action': True, 'label': 'run_code: eigenvalue trace theorem'},
    {'goal': 'Validate: the sum of the first N odd numbers always equals N². Test N=1 to 1000.',
     'expect_action': True, 'label': 'validate: odd numbers sum = N²'},
    {'goal': 'Implement three Fibonacci(35) methods: naive, memoized, iterative. Time each, print table.',
     'expect_action': True, 'label': 'run_code: Fibonacci benchmark'},
    {'goal': 'What is the halting problem and why is it significant in computer science?',
     'expect_action': False, 'label': 'Knowledge only: halting problem'},
    {'goal': 'Implement a BST with insert/search/delete/in-order. Insert [50,30,70,20,40,60,80,10,25,35,45], '
            'delete 30 and 70, print traversal, verify BST property.',
     'expect_action': True, 'label': 'run_code: BST with delete'},
    {'goal': 'Monte Carlo simulation to estimate pi with 1M points. Print estimate, error, 95% CI.',
     'expect_action': True, 'label': 'run_code: Monte Carlo pi'},
    {'goal': 'Implement Dijkstra from scratch (no networkx). 8 nodes, 12+ edges. Shortest path A→H.',
     'expect_action': True, 'label': 'run_code: Dijkstra from scratch'},
    {'goal': 'Combined benchmark: sieve(50000) + Monte Carlo(500k) + Dijkstra(10 nodes). '
            'Time each, print summary table in ms.',
     'expect_action': True, 'label': 'run_code: combined benchmark'},
]

action_results = []
t0_total = time.time()

for idx, turn in enumerate(ACTION_TURNS, 1):
    t0 = time.time()
    state = {**EMPTY_STATE, 'goal': turn['goal']}

    r = None
    for attempt in range(4):
        try:
            r = graph.invoke(state, config=action_config)
            break
        except Exception as exc:
            if '429' in str(exc) or 'RESOURCE_EXHAUSTED' in str(exc):
                wait = 20 * (attempt + 1)
                display(Markdown(f'> ⏳ Rate limit on T{idx} — retrying in {wait}s…'))
                time.sleep(wait)
            elif attempt < 3:
                time.sleep(5)
            else:
                display(Markdown(f'> ❌ T{idx} failed: `{str(exc)[:150]}`'))
                r = {'response': f'Error: {str(exc)[:200]}',
                     'needs_clarification': False, 'oriented_context': {},
                     'reasoning_trace': [], 'direction_result': {},
                     'action_result': {}, 'action_scratch': [],
                     'action_attempts': 0}

    elapsed = time.time() - t0
    dir_r = r.get('direction_result', {})
    act_r = r.get('action_result', {})
    scratch  = r.get('action_scratch', [])
    attempts = r.get('action_attempts', 0)
    ctx_r = r.get('oriented_context', {})
    turn_type = ctx_r.get('turn_type', dir_r.get('turn_type', '?'))
    conf = float(ctx_r.get('knowledge_confidence', 0.0))

    action_triggered = dir_r.get('needs_action', False)
    action_type   = dir_r.get('action_type', 'none')
    action_status = act_r.get('status', 'skipped')
    solution_stored = bool(act_r.get('solution_code', ''))

    route_ok = (turn['expect_action'] == action_triggered)

    if action_triggered:
        action_line = (f'🔧 `{action_type}` → `{action_status}` | '
                       f'Attempts: {attempts} | Scratch: {len(scratch)}')
        if solution_stored:
            action_line += ' | 💾 Stored'
    else:
        action_line = '📚 Pure knowledge path'

    if r.get('needs_clarification'):
        outcome = 'CLARIFICATION'
        body = '\n'.join(f'- {q}' for q in r.get('clarification_questions', []))
    else:
        outcome = 'ANSWERED'
        body = r.get('response', '*(no response)*')

    badge = '🟢' if outcome == 'ANSWERED' else '🟡'
    display(Markdown(
        f'---\n### T{idx}/12 {badge} {"✅" if route_ok else "❌"} *{turn["label"]}*\n\n'
        f'{action_line}\n\n'
        f'**Q:** *{turn["goal"][:200]}*\n\n{body}\n\n'
        f'> `turn={turn_type}` | `conf={conf:.2f}` | `{elapsed:.1f}s`'
    ))

    if len(scratch) > 1:
        rows = '\n'.join(
            f"| {e.get('attempt','?')} | {e.get('diagnosis','')[:80]} | "
            f"{(e.get('error','') or e.get('stderr',''))[:80]} |"
            for e in scratch
        )
        display(Markdown(f'<details><summary>🔍 Scratch pad ({len(scratch)} attempts)'
                         f'</summary>\n\n| Att | Diagnosis | Error |\n'
                         f'|----:|-----------|-------|\n{rows}\n</details>'))

    action_results.append(dict(
        t=idx, label=turn['label'], outcome=outcome,
        action_triggered=action_triggered, expect_action=turn['expect_action'],
        action_type=action_type, action_status=action_status,
        attempts=attempts, scratch_len=len(scratch),
        solution_stored=solution_stored, turn_type=turn_type,
        conf=conf, secs=elapsed,
    ))

total_elapsed = time.time() - t0_total
rows = '\n'.join(
    f"| {d['t']:>2} | {'✅' if d['expect_action']==d['action_triggered'] else '❌'} "
    f"| {d['label'][:35]:<35} | {d['action_type']:<10} | {d['action_status']:<8} "
    f"| {d['attempts']:>2} | {'💾' if d['solution_stored'] else '  '} "
    f"| {d['turn_type']:<12} | {d['conf']:>5.2f} | {d['secs']:>5.1f}s |"
    for d in action_results
)
routed_ok = sum(1 for d in action_results if d['expect_action'] == d['action_triggered'])
solutions_saved = sum(1 for d in action_results if d['solution_stored'])

display(Markdown(
    f'---\n## Action Agent Summary ⏱ {total_elapsed/60:.1f} min\n\n'
    f'**Routing:** {routed_ok}/12 | **Solutions stored:** {solutions_saved}\n\n'
    f'| # | OK | Label | Action | Status | Att | 💾 | Turn | Conf | Time |\n'
    f'|--:|:--:|-------|--------|--------|----:|:--:|------|-----:|------|\n{rows}'
))

---
### T1/12 🟢 ✅ *Knowledge only (no action)*

📚 Pure knowledge path

**Q:** *Explain the difference between a list and a tuple in Python. When should you use each?*

**Lists and Tuples in Python**
=====================================

In Python, lists and tuples are two fundamental data structures used to store collections of data. The primary difference between them lies in their mutability and usage guidelines.

**Mutability and Usage Guidelines**

* Lists are mutable, meaning they can be modified after creation [1]. They can store homogeneous data.
* Tuples are immutable, meaning they cannot be modified after creation [1]. They can store heterogeneous data.

**When to Use Each**

* Use lists when you need to modify the collection of data or when you need to store homogeneous data [1].
* Use tuples when you don't need to modify the collection of data and when you need to store heterogeneous data [2].

It's worth noting that while tuples are immutable, they can still be modified by reassigning a new tuple to the same variable. However, this does not change the original tuple object.

In general, it's a good practice to use lists for collections of data that need to be modified frequently and tuples for collections of data that do not need to be modified [2].

**Sources:**

* [Difference Between List and Tuple in Python - GeeksforGeeks](https://www.geeksforgeeks.org/python/python-difference-between-list-and-tuple/)
* [Lists vs Tuples in Python](https://realpython.com/python-lists-tuples/)
* [python - List vs tuple, when to use each? - Stack Overflow](https://stackoverflow.com/questions/1708510/list-vs-tuple-when-to-use-each)
* [Python Lists vs Tuples: Understanding Their Differences and ... - Udacity](https://www.udacity.com/blog/2025/02/python-lists-vs-tuples-understanding-their-differences-and-best-uses.html)
* [Python Lists vs Tuples: Key Differences Explained - PyTutorial](https://pytutorial.com/python-lists-vs-tu

> `turn=new_topic` | `conf=0.86` | `26.0s`

---
### T2/12 🟢 ✅ *check_env: runtime inspection*

🔧 `validate` → `success` | Attempts: 3 | Scratch: 3

**Q:** *Check what Python version is installed, and list all numpy/scipy/pandas versions available.*

=== Topic transition ===
We were discussing **explain_the_difference**; your question now shifts to a new topic — starting fresh.

To check the Python version installed, you can use `python --version` in your terminal or import the `sys` module in Python and use `sys.version` or `sys.version_info` to get the version information [1]. 

For NumPy, SciPy, and Pandas versions, you can use the following methods:
- Use `import numpy as np; print(np.__version__)`, `import scipy as sp; print(sp.__version__)`, or `import pandas as pd; print(pd.show_versions())` in your Python code [2].
- Alternatively, you can use `pkg_resources.get_distribution('numpy').version`, `pkg_resources.get_distribution('scipy').version`, and `pkg_resources.get_distribution('pandas').version` to get the version information of these packages.

Note that these methods will provide you with the version information of the packages installed on your system. 

**Python Version:** 3.12.13
**NumPy Version:** 2.0.2 (latest version unknown)
**SciPy Version:** 1.16.3 (latest version unknown)
**Pandas Version:** 2.2.2 (latest version unknown)

> `turn=new_topic` | `conf=0.76` | `61.6s`

<details><summary>🔍 Scratch pad (3 attempts)</summary>

| Att | Diagnosis | Error |
|----:|-----------|-------|
| 1 | The code is trying to import a non-existent package called 'pkg'. This package d |  |
| 2 | The code is trying to import numpy/scipy/pandas versions but it seems that the ' |  |
| 3 | Attempted to fix the issue by using pkg_resources instead of direct import. |  |
</details>

---
### T3/12 🟢 ❌ *run_code: Sieve of Eratosthenes*

📚 Pure knowledge path

**Q:** *Write and run a Sieve of Eratosthenes to find primes below 10000. Print count and last 10.*

=== Topic transition ===
We were discussing **check_what_python**; your question now shifts to a new topic — starting fresh.

Here is the Sieve of Eratosthenes algorithm implemented in Python:

```python
def sieve_of_eratosthenes(n):
    prime = [True] * (n + 1)
    p = 2
    while (p * p <= n):
        if prime[p]:
            for i in range(p * p, n + 1, p):
                prime[i] = False
        p += 1

    return [p for p in range(2, n) if prime[p]]

primes = sieve_of_eratosthenes(10000)

print("Count:", len(primes))
print("Last 10 Primes:")
for i in range(max(0, len(primes) - 11), len(primes)):
    print(primes[i])
```

This code uses the Sieve of Eratosthenes algorithm to find and print prime numbers below a specified limit (in this case, 10000). The `sieve_of_eratosthenes` function takes an integer `n` as input and returns a list of all prime numbers in the range [2, n]. The main part of the code calls this function with `n = 10000`, prints the count of primes found, and then prints the last 10 primes.

> `turn=new_topic` | `conf=0.84` | `43.9s`

<details><summary>🔍 Scratch pad (3 attempts)</summary>

| Att | Diagnosis | Error |
|----:|-----------|-------|
| 1 | The code is trying to import a non-existent package called 'pkg'. This package d |  |
| 2 | The code is trying to import numpy/scipy/pandas versions but it seems that the ' |  |
| 3 | Attempted to fix the issue by using pkg_resources instead of direct import. |  |
</details>

...

---
### T7/12 🟢 ❌ *run_code: Fibonacci benchmark*

📚 Pure knowledge path

**Q:** *Implement three Fibonacci(35) methods: naive, memoized, iterative. Time each, print table.*

=== Topic transition ===
We were discussing **validate:_the_sum**; your question now shifts to a new topic — starting fresh.



### Implementing Fibonacci Methods with Timing

Below are the three Fibonacci methods implemented in Python: naive, memoized, and iterative.

#### Prerequisites:

*   You have basic knowledge of Python programming.
*   You have the necessary libraries installed (in this case, none).

#### Steps to implement the Fibonacci methods with timing:

1.  **Naive Method**
    *   The naive method calculates each Fibonacci number recursively from scratch.
    *   It is inefficient for large values of `n` because it performs a lot of redundant calculations.

    ```python
def fibonacci_naive(n):
    if n <= 0:
        return "Input should be a positive integer."
    elif n == 1:
        return 0
    elif n == 2:
        return 1
    else:
        fib_sequence = [0, 1]
        while len(fib_sequence) < n:
            fib_sequence.append(fib_sequence[-1] + fib_sequence[-2])
        return fib_sequence[-1]
```

2.  **Memoized Method**
    *   The memoized method uses a dictionary to store previously calculated Fibonacci numbers.
    *   It is more efficient than the naive method because it avoids redundant calculations.

    ```python
def fibonacci_memoized(n):
    memo = {0: 0, 1: 1}
    def fib(n):
        if n not in memo:
            memo[n] = fib(n-1) + fib(n-2)
        return memo[n]
    return fib(n)
```

3.  **Iterative Method**
    *   The iterative method uses a loop to calculate each Fibonacci number.
    *   It is the most efficient of the three methods because it avoids function calls and recursive calculations.

    ```python
def fibonacci_iterative(n):
    a, b = 0, 1
    for _ in range(2, n+1):
        a, b = b, a+b
    return b
```

#### Timing Each Method

To time each method, we use the `time` module to measure the execution time.

```python
import time

n = 35

start_time = time.time()
print("Naive approach:", fibonacci_naive(n))
end_time = time.time()
naive_time = end_time - start_time

start_time = time.time()
print("Memoized approach:", fibonacci_memoized(n))
end_time = time.time()
memoized_time = end_time - start_time

start_time = time.time()
print("Iterative approach:", fibonacci_iterative(n))
end_time = time.time()
iterative_time = end_time - start_time
```

#### Printing the Results

Finally, we print the results in a table format.

```python
print("\nTime taken by each method:")
print(f"Naive: {naive_time} seconds")
print(f"Memoized: {memoized_time} seconds")
print(f"Iterative: {iterative_time} seconds")
```

Note that the actual time taken by each method may vary depending on your system's performance.

Sources:

[1] (relevance: 0.786)
Summary: This text describes the implementation of three different methods to calculate the 35th Fibonacci number: naive, memoized, and iterative.
Content: Based on the search results, I will implement three Fibonacci(35) methods: naive, memoized, and iterative.

[2] (relevance: 0.735)
Summary: This code compares the execution times of three different algorithms for calculating the nth Fibonacci number: naive, memoized, and iterative.
Content:  approach:

[3] (relevance: 0.673)
Summary: The text discusses the limitations of a computational method called Variational Quantum Eigensolver (VQE) and explores alternative approaches to address these limitations.
Content: nvergence rate is often slow, requiring many iterations to achieve accurate results. This can be time-consuming and may not be feasible for larger systems.

> `turn=new_topic` | `conf=0.79` | `37.9s`

<details><summary>🔍 Scratch pad (15 attempts)</summary>

| Att | Diagnosis | Error |
|----:|-----------|-------|
| 1 | The code is trying to import a non-existent package called 'pkg'. This package d |  |
| 2 | The code is trying to import numpy/scipy/pandas versions but it seems that the ' |  |
| 3 | Attempted to fix the issue by using pkg_resources instead of direct import. |  |
| 1 |  |  |
| 2 | First attempt. |  |
| 1 | The code is trying to verify if the trace of a random matrix equals the sum of i |  |
| 2 | The code is trying to verify if the trace of a random matrix equals the sum of i |  |
| 3 |  |  |
| 4 | The code is trying to verify if the trace of a random matrix equals the sum of i |  |
| 5 |  |  |
| 1 |  |  |
| 2 |  |  |
| 3 |  |  |
| 4 |  |  |
| 5 |  |  |
</details>

---
### T8/12 🟢 ✅ *Knowledge only: halting problem*

📚 Pure knowledge path

**Q:** *What is the halting problem and why is it significant in computer science?*

=== The halting problem ===
The halting problem, proven by Alan Turing in 1936, is undecidable, meaning there cannot exist an algorithm that can determine whether any given program will halt for all possible inputs [1]. This fundamental concept in computability theory highlights the limitations of algorithms and the importance of understanding what can and cannot be computed.

The significance of the halting problem lies in its implications for programming and computer science as a whole. It shows that there are limits to what can be computed and that some problems are inherently unsolvable [2]. The halting problem has far-reaching consequences, including limitations on the power of algorithms and the importance of understanding the trade-offs between different approaches.

**Sources:**

[1] (relevance: 0.868)
Summary: This text discusses the halting problem, a concept in computability theory that explains why certain problems are undecidable by any algorithm.
Content: e all possible decision problems.
*   Undecidability: Many problems in computer science are undecidable, meaning they cannot be solved by any algorithm.
*   Implications for programming languages: The halting problem has implications for the design of programming languages and the development of compilers.

[2] (relevance: 0.849)
Summary: This text is about the halting problem in computability theory, a concept proven by Alan Turing to be undecidable, with significant implications for programming and computer science.
Content: The halting problem is a fundamental concept in computability theory, which deals with determining whether a computer program will halt (terminate) or run indefinitely when executed with a given input. Alan Turing proved in 1936 that the halting problem is undecidable, meaning there cannot exist an algorithm that can determine whether any given program will halt for all possible inputs.

[3] (relevance: 0.658)
Summary: The text discusses the limitations of a computational method called Variational Quantum Eigensolver (VQE) and explores alternative approaches to address these limitations.
Content: nvergence rate is often slow, requiring many iterations to achieve accurate results. This can be time-consuming and may not be feasible for larger systems.

Note: I did not include [3] in the answer as its relevance is lower than the other two sources and it does not directly relate to the halting problem.

> `turn=new_topic` | `conf=0.87` | `25.5s`

<details><summary>🔍 Scratch pad (15 attempts)</summary>

| Att | Diagnosis | Error |
|----:|-----------|-------|
| 1 | The code is trying to import a non-existent package called 'pkg'. This package d |  |
| 2 | The code is trying to import numpy/scipy/pandas versions but it seems that the ' |  |
| 3 | Attempted to fix the issue by using pkg_resources instead of direct import. |  |
| 1 |  |  |
| 2 | First attempt. |  |
| 1 | The code is trying to verify if the trace of a random matrix equals the sum of i |  |
| 2 | The code is trying to verify if the trace of a random matrix equals the sum of i |  |
| 3 |  |  |
| 4 | The code is trying to verify if the trace of a random matrix equals the sum of i |  |
| 5 |  |  |
| 1 |  |  |
| 2 |  |  |
| 3 |  |  |
| 4 |  |  |
| 5 |  |  |
</details>

: 

## Test D — Physics 12-turn conversation

General relativity → gravitational waves → black holes → dark matter → synthesis.  
Tests memory continuity, coverage accumulation, topic pivots, and synthesis.

In [ ]:
import time
from IPython.display import display, Markdown

PHYSICS_TURNS = [
    'Explain the theory of general relativity and what it changed about our understanding of gravity.',
    'How does general relativity differ from Newton\'s theory of gravity and from special relativity?',
    'What are gravitational waves and how were they first detected?',
    'Who were the key scientists and institutions behind the LIGO gravitational wave detection?',
    'How does GPS rely on both special and general relativity to stay accurate to within metres?',
    'What is a black hole and how do they form from dying stars?',
    'What happens at the event horizon of a black hole — can anything escape?',
    'What is Hawking radiation and why does it suggest black holes eventually evaporate?',
    'How are supermassive black holes connected to galaxy formation and active galactic nuclei?',
    'What is dark matter? What observational evidence do we have for its existence?',
    'What is dark energy and how does it explain the accelerating expansion of the universe?',
    'Summarise the key unsolved mysteries in modern physics that we have discussed.',
]

conv_config = {'configurable': {'thread_id': thread_id}}
physics_results = []
total_t0 = time.time()

for idx, goal in enumerate(PHYSICS_TURNS, 1):
    t0 = time.time()
    state = {**EMPTY_STATE, 'goal': goal}

    for attempt in range(4):
        try:
            r = graph.invoke(state, config=conv_config)
            break
        except Exception as e:
            if '429' in str(e) or 'RESOURCE_EXHAUSTED' in str(e):
                wait = 15 * (attempt + 1)
                display(Markdown(f'> ⏳ Rate limit — retrying in {wait}s…'))
                time.sleep(wait)
            else:
                raise

    elapsed   = time.time() - t0
    ctx_r     = r.get('oriented_context', {})
    turn_type = ctx_r.get('turn_type', '?')
    coverage  = ctx_r.get('coverage', '?')
    conf      = ctx_r.get('knowledge_confidence', 0)

    if r.get('needs_clarification'):
        outcome = 'CLARIFICATION'
        body = '\n'.join(f'- {q}' for q in r.get('clarification_questions', []))
    else:
        outcome = 'ANSWERED'
        body    = r.get('response', '*(no response)*')

    badge = '🟢' if outcome == 'ANSWERED' else '🟡'
    display(Markdown(
        f'---\n### Turn {idx}/{len(PHYSICS_TURNS)} {badge} {outcome}\n'
        f'**Q:** *{goal}*\n\n{body}\n\n'
        f'> `turn={turn_type}` | `coverage={coverage}` | '
        f'`conf={conf:.2f}` | `{elapsed:.1f}s`'
    ))

    physics_results.append(dict(
        q=idx, outcome=outcome, turn_type=turn_type,
        coverage=coverage, conf=f'{conf:.2f}', time_s=f'{elapsed:.1f}',
    ))

total_elapsed = time.time() - total_t0
rows = '\n'.join(
    f"| {d['q']:>2} | {d['outcome']:<15} | {d['turn_type']:<12} | {d['coverage']:<9} "
    f"| {d['conf']:>5} | {d['time_s']:>5}s |"
    for d in physics_results
)
display(Markdown(
    f'---\n## Physics summary — {total_elapsed:.0f}s total\n\n'
    f'| # | Outcome | Turn type | Coverage | Conf | Time |\n'
    f'|--:|---------|-----------|----------|-----:|------|\n{rows}'
))

## Test E — Quantum 30-turn conversation

Quantum Computing (Q1-16) → QM Interpretations (Q17-23) → Comeback (Q24-26) →
Sensing/Networks (Q27-29) → Synthesis (Q30).  
Generates a confidence trend chart with turn-type classification.

In [ ]:
import time, uuid
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

q30_thread = str(uuid.uuid4())
q30_config = {'configurable': {'thread_id': q30_thread}}

Q30_TURNS = [
    # Foundations (Q1-7)
    'What is a qubit and how does superposition make it different from a classical bit?',
    'Explain quantum entanglement — what it is, how it is created, and why Einstein called it spooky action at a distance.',
    'What is quantum decoherence, why does it destroy quantum states, and what timescales matter in superconducting hardware?',
    'Walk me through quantum gates — explain Hadamard, CNOT, and Toffoli gates.',
    'How does Shor\'s algorithm factor large integers faster than classical computers?',
    'Explain Grover\'s algorithm — what quadratic speedup does it achieve?',
    'What is quantum error correction? Why can\'t classical error-correction ideas be directly applied?',
    # Hardware (Q8-12)
    'Explain the surface code: logical qubits, stabilizers, syndrome measurement, and error detection.',
    'What is the fault-tolerance threshold theorem? What gate error rates do current processors achieve?',
    'Compare superconducting qubits, trapped ions, and photonic approaches — tradeoffs in coherence, fidelity, scalability.',
    'What is the NISQ era? What algorithms work on noisy 100-1000 qubit devices?',
    'Assess Google Sycamore\'s quantum supremacy claim (2019) and counter-arguments.',
    # Advanced (Q13-16)
    'What are topological qubits? Explain non-Abelian anyons and braiding operations.',
    'How does the variational quantum eigensolver (VQE) work and what are its NISQ limitations?',
    'What is quantum volume as a hardware benchmark?',
    'What is the current (2025) state of quantum computing — milestones, roadmaps, timeline to fault tolerance?',
    # Topic change: QM foundations (Q17-23)
    'Setting quantum computers aside — what is the measurement problem in quantum mechanics?',
    'Compare Copenhagen, Many-Worlds, and de Broglie–Bohm pilot-wave theory.',
    'How does decoherence explain apparent wave function collapse? Why do critics say it\'s incomplete?',
    'What is Rovelli\'s relational interpretation and how does QBism differ?',
    'What does Bell\'s theorem prove? How do loophole-free Bell tests constrain local hidden variables?',
    'Is the wave function real (ψ-ontic) or epistemic? What does the PBR theorem imply?',
    'Could QM be emergent from a deeper deterministic substrate?',
    # Comeback (Q24-26)
    'Returning to quantum computing — how do the measurement problem and decoherence bear on error correction?',
    'What is the quantum threat to cryptography? When could a fault-tolerant QC break RSA-2048?',
    'Which post-quantum crypto algorithms has NIST standardised? How does lattice hardness resist quantum attacks?',
    # Sensing/Networks (Q27-29)
    'What is quantum sensing? How do atomic clocks and NV magnetometers exploit quantum coherence?',
    'How does QKD work? Explain BB84 and the no-cloning theorem.',
    'What is a quantum repeater and why is it needed for a global quantum internet?',
    # Synthesis (Q30)
    'Synthesise our full conversation: qubits, error correction, QM interpretations, crypto, sensing. What are the three deepest open questions?',
]
assert len(Q30_TURNS) == 30

SECTIONS = {1: 'Foundations', 8: 'Hardware', 13: 'Advanced',
            17: 'Topic change', 24: 'Comeback', 27: 'Sensing', 30: 'Synthesis'}
_SECT_STYLE = {
    1:  dict(color='steelblue',  lw=1.0, ls='--'),
    8:  dict(color='slategray',  lw=1.0, ls='--'),
    13: dict(color='slategray',  lw=1.0, ls='--'),
    17: dict(color='darkorange', lw=2.2, ls='-'),
    24: dict(color='purple',     lw=2.2, ls='-'),
    27: dict(color='slategray',  lw=1.0, ls='--'),
    30: dict(color='slategray',  lw=1.0, ls='--'),
}

q30_results = []
t0_total = time.time()

for idx, goal in enumerate(Q30_TURNS, 1):
    t0 = time.time()
    state = {**EMPTY_STATE, 'goal': goal}

    r = {'response': 'Error: max retries', 'needs_clarification': False,
         'oriented_context': {}, 'reasoning_trace': []}
    for attempt in range(4):
        try:
            r = graph.invoke(state, config=q30_config)
            break
        except Exception as exc:
            if '429' in str(exc) or 'RESOURCE_EXHAUSTED' in str(exc):
                wait = 20 * (attempt + 1)
                display(Markdown(f'> ⏳ Rate limit Q{idx} — retrying in {wait}s…'))
                time.sleep(wait)
            elif attempt < 3:
                time.sleep(5)
            else:
                display(Markdown(f'> ❌ Q{idx} failed: `{str(exc)[:120]}`'))

    elapsed   = time.time() - t0
    ctx_r     = r.get('oriented_context', {})
    turn_type = ctx_r.get('turn_type', '?')
    coverage  = ctx_r.get('coverage', '?')
    conf      = float(ctx_r.get('knowledge_confidence', 0.0))
    section   = next(SECTIONS[s] for s in sorted(SECTIONS, reverse=True) if idx >= s)

    if r.get('needs_clarification'):
        outcome = 'CLARIFICATION'
        body = '\n'.join(f'- {q}' for q in r.get('clarification_questions', []))
    else:
        outcome = 'ANSWERED'
        body    = r.get('response', '*(no response)*')

    badge = '🟢' if outcome == 'ANSWERED' else '🟡'
    display(Markdown(
        f'---\n### Q{idx}/30 {badge} {outcome} · *{section}*\n'
        f'**Q:** *{goal}*\n\n{body}\n\n'
        f'> `turn={turn_type}` | `coverage={coverage}` | '
        f'`conf={conf:.2f}` | `{elapsed:.1f}s`'
    ))
    q30_results.append(dict(n=idx, outcome=outcome, turn_type=turn_type,
                            coverage=coverage, conf=conf, secs=elapsed))

elapsed_total = time.time() - t0_total

# Summary table
rows = '\n'.join(
    f"| {d['n']:>2} | {'🟢' if d['outcome']=='ANSWERED' else '🟡'} {d['outcome']:<14} "
    f"| {d['turn_type']:<12} | {d['coverage']:<9} | {d['conf']:>5.2f} | {d['secs']:>5.1f}s |"
    for d in q30_results
)
display(Markdown(
    f'---\n## 30-turn summary ⏱ {elapsed_total/60:.1f} min\n\n'
    f'| # | Outcome | Turn type | Coverage | Conf | Time |\n'
    f'|--:|---------|-----------|----------|-----:|------|\n{rows}'
))

# ── Confidence trend chart ────────────────────────────────────────────
turns_x = [d['n']    for d in q30_results]
confs_y = [d['conf'] for d in q30_results]
window  = 4
rolling = [sum(confs_y[max(0,i-window+1):i+1]) / min(window,i+1)
           for i in range(len(confs_y))]

outcome_colors = ['#2ecc71' if d['outcome']=='ANSWERED' else '#e67e22'
                  for d in q30_results]
type_palette = {'new_topic': '#e74c3c', 'follow_up': '#2ecc71',
                'elaboration': '#3498db', 'clarification': '#9b59b6',
                'correction': '#f39c12', '?': '#95a5a6'}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(17, 9),
                                gridspec_kw={'height_ratios': [3, 1]})

ax1.fill_between(turns_x, confs_y, alpha=0.10, color='steelblue')
(l1,) = ax1.plot(turns_x, confs_y, color='steelblue', lw=1.5, alpha=0.65,
                  label='Per-turn confidence')
(l2,) = ax1.plot(turns_x, rolling, color='royalblue', lw=2.5,
                  label=f'{window}-turn rolling avg')
ax1.scatter(turns_x, confs_y, c=outcome_colors, s=70, zorder=6,
            edgecolors='white', linewidths=0.5)
(l3,) = ax1.plot([], [], color='gray', ls=':', lw=1.2, alpha=0.8,
                  label='Memory threshold (0.65)')
ax1.axhline(0.65, color='gray', ls=':', lw=1.2, alpha=0.8)

for q, lbl in SECTIONS.items():
    st = _SECT_STYLE.get(q, dict(color='gray', lw=1, ls='--'))
    if q > 1:
        ax1.axvline(q - 0.5, color=st['color'], ls=st['ls'], lw=st['lw'], alpha=0.75)
    ax1.text(q + 0.1, 1.05, lbl, fontsize=7.5, color=st['color'],
             rotation=40, va='bottom', ha='left')

ax1.set_xlim(0.5, 30.7); ax1.set_ylim(0, 1.30)
ax1.set_xticks(range(1, 31))
ax1.set_xlabel('Turn #'); ax1.set_ylabel('Knowledge confidence')
ax1.set_title('Knowledge Confidence Trend — 30 turns', fontweight='bold')
ax1.legend(
    handles=[mpatches.Patch(color='#2ecc71', label='🟢 ANSWERED'),
             mpatches.Patch(color='#e67e22', label='🟡 CLARIFICATION'),
             l1, l2, l3],
    loc='upper left', fontsize=9, ncol=2)
ax1.grid(True, alpha=0.18)

for d in q30_results:
    ax2.bar(d['n'], 1, color=type_palette.get(d['turn_type'], '#95a5a6'),
            alpha=0.85, width=0.8)
for q in SECTIONS:
    if q > 1:
        st = _SECT_STYLE.get(q, dict(color='gray', lw=1, ls='--'))
        ax2.axvline(q - 0.5, color=st['color'], ls=st['ls'], lw=st['lw'], alpha=0.70)
ax2.set_xlim(0.5, 30.7); ax2.set_ylim(0, 1.1)
ax2.set_xticks(range(1, 31))
ax2.set_xlabel('Turn #'); ax2.set_yticks([])
ax2.set_title('Turn Classification', fontsize=10)
ax2.legend(handles=[mpatches.Patch(color=c, label=t)
                     for t, c in type_palette.items() if t != '?'],
           loc='upper right', fontsize=8, ncol=5)
ax2.grid(True, alpha=0.15, axis='x')

plt.tight_layout(h_pad=2.5)
chart_path = '/content/Brain/quantum_confidence_trend.png'
plt.savefig(chart_path, dpi=130, bbox_inches='tight')
plt.show()
display(Markdown(f'Chart saved → `{chart_path}`'))